# YOLOv11n baseline on tank_qwen_v2 (400 synthetic frames)

Goal — see how a stock YOLO handles our synthetic tank dataset out of the box.

**Setup:**
- YOLOv11n (smallest, fastest iter)
- `imgsz=1280` — matches inference strategy (SAHI + P2 head recipe; distant tanks are ~10 px on 640, so we train at higher res)
- `batch=4` (T4 16 GB fits this at 1280)
- 50 epochs, stock augmentations (mosaic + mixup on by default)
- Stratified 70/20/10 split by distance zone (near ≤200 m, mid 300-500 m, far 600-800 m)

## 1. Attach dataset

In [ ]:
# Kaggle mounts CLI-attached datasets at a different path than UI-attached ones,
# so auto-detect where the folder ended up.
from pathlib import Path
candidates = list(Path('/kaggle/input').rglob('data.yaml'))
assert candidates, 'no data.yaml found under /kaggle/input — is the dataset attached?'
SRC = candidates[0].parent
print('dataset root:', SRC)
for p in sorted(SRC.iterdir()):
    print(' ', p.name)

## 2. Install ultralytics

In [ ]:
!pip install -q --upgrade ultralytics
import ultralytics
ultralytics.checks()

## 3. Materialise dataset in `/kaggle/working` with absolute paths

In [ ]:
import shutil
from pathlib import Path

# SRC comes from cell 1 (auto-detected). Fallback if this cell is run alone.
try:
    SRC  # noqa: F821
except NameError:
    SRC = next(Path('/kaggle/input').rglob('data.yaml')).parent

DST = Path('/kaggle/working/dataset')
if DST.exists():
    shutil.rmtree(DST)
DST.mkdir(parents=True)

for sub in ('images', 'labels'):
    shutil.copytree(SRC / sub, DST / sub)

(DST / 'data.yaml').write_text(f'''path: {DST}
train: images/train
val: images/val
test: images/test
names:
  0: tank
''')

print('splits:')
for split in ('train', 'val', 'test'):
    n = len(list((DST / 'images' / split).glob('*.png')))
    print(f'  {split}: {n}')

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='/kaggle/working/dataset/data.yaml',
    imgsz=1280,
    batch=4,
    epochs=50,
    device=0,
    project='/kaggle/working/runs',
    name='tank_qwen_v2_baseline',
    exist_ok=True,
    patience=15,
    cache=False,
    workers=2,
    seed=42,
    plots=True,
)

## 5. Evaluate on test split

In [ ]:
metrics = model.val(
    data='/kaggle/working/dataset/data.yaml',
    split='test',
    imgsz=1280,
    batch=4,
    plots=True,
)
print('mAP50   :', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('per-class mAP:', metrics.box.maps)

## 6. Show sample predictions

In [ ]:
import random
from pathlib import Path
from IPython.display import Image, display

test_imgs = sorted(Path('/kaggle/working/dataset/images/test').glob('*.png'))
random.seed(0)
picks = random.sample(test_imgs, k=min(6, len(test_imgs)))

for p in picks:
    r = model.predict(source=str(p), imgsz=1280, conf=0.25, save=True,
                      project='/kaggle/working/preds', name='test', exist_ok=True)
    saved = Path(r[0].save_dir) / p.name
    print(p.name, '→', r[0].boxes.shape[0], 'detections')
    if saved.exists():
        display(Image(filename=str(saved)))

## 7. Package best weights for download

In [ ]:
import shutil
from pathlib import Path

run_dir = Path('/kaggle/working/runs/tank_qwen_v2_baseline')
best = run_dir / 'weights' / 'best.pt'
print('best.pt:', best, 'exists:', best.exists())

# Copy to /kaggle/working root so it shows up in the notebook output panel.
if best.exists():
    shutil.copy2(best, '/kaggle/working/tank_qwen_v2_yolo11n_best.pt')

for name in ('results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
             'labels.jpg', 'PR_curve.png', 'val_batch0_pred.jpg'):
    src = run_dir / name
    if src.exists():
        shutil.copy2(src, f'/kaggle/working/{name}')
        print('copied', name)

## 8. Per-zone evaluation

Split test set by distance zone (near ≤200 m / mid 300-500 m / far 600-800 m) and run `model.val()` on each subset. The `all` line reveals how mAP degrades with distance.

In [ ]:
import shutil
from pathlib import Path

ZONES = {
    "near": ["tank_00047", "tank_00063", "tank_00120", "tank_00202", "tank_00237",
             "tank_00251", "tank_00266", "tank_00318", "tank_00351", "tank_00358",
             "tank_00386", "tank_00395"],
    "mid":  ["tank_00006", "tank_00015", "tank_00024", "tank_00070", "tank_00099",
             "tank_00163", "tank_00206", "tank_00228", "tank_00277", "tank_00315",
             "tank_00378", "tank_00380"],
    "far":  ["tank_00019", "tank_00022", "tank_00025", "tank_00065", "tank_00069",
             "tank_00075", "tank_00083", "tank_00104", "tank_00158", "tank_00164",
             "tank_00169", "tank_00175", "tank_00189", "tank_00279", "tank_00332",
             "tank_00346"],
}

DST = Path('/kaggle/working/dataset')  # from cell 3
zone_results = {}

for zone, stems in ZONES.items():
    zdir = Path(f'/kaggle/working/dataset_{zone}')
    if zdir.exists():
        shutil.rmtree(zdir)
    (zdir / 'images' / 'test').mkdir(parents=True)
    (zdir / 'labels' / 'test').mkdir(parents=True)

    for stem in stems:
        img = DST / 'images' / 'test' / f'{stem}.png'
        lbl = DST / 'labels' / 'test' / f'{stem}.txt'
        if img.exists():
            shutil.copy2(img, zdir / 'images' / 'test' / img.name)
        if lbl.exists():
            shutil.copy2(lbl, zdir / 'labels' / 'test' / lbl.name)

    (zdir / 'data.yaml').write_text(f'''path: {zdir}
train: images/test
val: images/test
test: images/test
names:
  0: tank
''')

    n = len(list((zdir / 'images' / 'test').glob('*.png')))
    print(f'\n=== zone: {zone} ({n} frames) ===')
    m = model.val(
        data=str(zdir / 'data.yaml'),
        split='test',
        imgsz=1280,
        batch=4,
        plots=False,
        verbose=False,
    )
    zone_results[zone] = {
        'n': n,
        'P': float(m.box.mp),
        'R': float(m.box.mr),
        'mAP50': float(m.box.map50),
        'mAP50-95': float(m.box.map),
    }

print('\n\n=== per-zone summary ===')
print(f"{'zone':<8} {'n':>4} {'P':>7} {'R':>7} {'mAP50':>7} {'mAP50-95':>9}")
for zone, r in zone_results.items():
    print(f"{zone:<8} {r['n']:>4} {r['P']:>7.3f} {r['R']:>7.3f} {r['mAP50']:>7.3f} {r['mAP50-95']:>9.3f}")

import json
Path('/kaggle/working/per_zone_metrics.json').write_text(json.dumps(zone_results, indent=2))
print('\nsaved /kaggle/working/per_zone_metrics.json')